In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from sova.bench_matmul import matmul_tflops, time_matmul

print(sys.executable)
print("numpy", np.__version__, "| torch", torch.__version__)

In [ ]:
a = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])
print(a.shape, a.dtype, a.ndim)
print(a * 10)
print(a.sum(), a.sum(axis=0), a.sum(axis=1))

In [ ]:
row = np.array([100.0, 200.0, 300.0])
print(a + row)
col = np.array([[1.0], [2.0]])
print(a + col)

In [ ]:
x = np.random.default_rng(0).random(1_000_000)


def sum_squares_loop(values):
    total = 0.0
    for v in values:
        total += v * v
    return total


print(sum_squares_loop(x), np.dot(x, x))

In [ ]:
%timeit sum_squares_loop(x)
%timeit np.dot(x, x)

In [ ]:
sizes = [256, 512, 1024, 2048, 4096]
cases = [("cpu", torch.float32)]
if torch.cuda.is_available():
    sizes.append(8192)
    cases += [("cuda", torch.float32), ("cuda", torch.bfloat16)]

results = {}
for device, dtype in cases:
    speeds = []
    for n in sizes:
        if device == "cpu" and n > 4096:
            speeds.append(np.nan)
            continue
        seconds = time_matmul(device, dtype, n=n, repeats=5)
        speeds.append(matmul_tflops(n, seconds))
    label = f"{device} {str(dtype).removeprefix('torch.')}"
    results[label] = speeds
    print(label, [round(s, 2) for s in speeds])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, speeds in results.items():
    ax.plot(sizes, speeds, marker="o", label=label)
ax.set_xscale("log", base=2)              # размеры растут вдвое — равные шаги по оси
ax.set_xticks(sizes, [str(n) for n in sizes])
ax.set_xlabel("размер матрицы N")
ax.set_ylabel("TFLOPS")
ax.set_title("Скорость умножения N×N на N×N")
ax.grid(True, alpha=0.3)
ax.legend()

runs = Path("..") / "runs"                # папка runs/ в .gitignore — картинка не попадёт в Git
runs.mkdir(exist_ok=True)
fig.savefig(runs / "04_tflops.png", dpi=150, bbox_inches="tight")
plt.show()

При маленьких размерах различий почти не видно.
Основной скачок у cuda bfloat16 происходит между 512 и 2048 (с ~7 до ~55 tflops).
Основной скачок у cuda float32 происходит между 4096 и 8192 (с ~4 до ~19 tflops).
У cpu при больших числах наоборот число tflops падает.
В пике cuda bfloat16 быстрее cpu примерно в 70 раз, что очень впечатляюще.